In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import copy

# 1. The Global Model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2)

    def forward(self, x):
        return self.fc(x)

# 2. The Client Class
class Client:
    def __init__(self, client_id, model, dataloader, lr=0.01):
        self.id = client_id
        self.local_model = copy.deepcopy(model)
        self.dataloader = dataloader
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.SGD(self.local_model.parameters(), lr=lr)

    def train(self, epochs=1):
        self.local_model.train()
        for epoch in range(epochs):
            for data, target in self.dataloader:
                self.optimizer.zero_grad()
                output = self.local_model(data)
                loss = self.criterion(output, target)
                loss.backward()
                self.optimizer.step()

        num_samples = len(self.dataloader.dataset)
        return num_samples, self.local_model.state_dict()

# 3. Server Aggregation Logic (Assignment 5 Core)
def server_aggregate(global_model, client_updates):
    print("\n--- Server Aggregation Process ---")
    total_samples = sum([num for num, _ in client_updates])
    print(f"Total samples across all clients: {total_samples}")

    # Initialize a dictionary to hold the aggregated weights
    aggregated_weights = copy.deepcopy(client_updates[0][1])
    for key in aggregated_weights.keys():
        aggregated_weights[key] = torch.zeros_like(aggregated_weights[key])

    # Perform weighted averaging
    for client_id, (num_samples, client_state_dict) in enumerate(client_updates):
        weight_fraction = num_samples / total_samples
        print(f"Applying Client {client_id + 1} weights (Weight Fraction: {weight_fraction:.2f})")
        for key in aggregated_weights.keys():
            aggregated_weights[key] += client_state_dict[key] * weight_fraction

    # Update the global model
    global_model.load_state_dict(aggregated_weights)
    print("Global model successfully updated with aggregated weights.")
    return global_model

# ==========================================
# EXECUTION SCRIPT
# ==========================================
if __name__ == "__main__":
    print("--- Starting Assignment 5 Simulation ---\n")

    global_model = SimpleModel()

    # Store the initial weights of the global model to compare later
    initial_weights = copy.deepcopy(global_model.state_dict()['fc.weight'])
    print("Initial Global Model Weights (first 3 values):", initial_weights[0][:3].tolist())

    # Create Client 1 (100 samples)
    X1, y1 = torch.randn(100, 10), torch.randint(0, 2, (100,))
    loader1 = DataLoader(TensorDataset(X1, y1), batch_size=10, shuffle=True)
    client_1 = Client(1, global_model, loader1)

    # Create Client 2 (200 samples - This client will have 2x the weight in FedAvg)
    X2, y2 = torch.randn(200, 10), torch.randint(0, 2, (200,))
    loader2 = DataLoader(TensorDataset(X2, y2), batch_size=10, shuffle=True)
    client_2 = Client(2, global_model, loader2)

    # Train both clients locally
    print("\nTraining clients locally...")
    updates = []
    updates.append(client_1.train(epochs=3))
    updates.append(client_2.train(epochs=3))

    # Server aggregates the updates
    global_model = server_aggregate(global_model, updates)

    # Show the difference
    final_weights = global_model.state_dict()['fc.weight']
    print("\nFinal Aggregated Global Weights (first 3 values):", final_weights[0][:3].tolist())

--- Starting Assignment 5 Simulation ---

Initial Global Model Weights (first 3 values): [-0.19407136738300323, 0.26817330718040466, 0.2542476952075958]

Training clients locally...

--- Server Aggregation Process ---
Total samples across all clients: 300
Applying Client 1 weights (Weight Fraction: 0.33)
Applying Client 2 weights (Weight Fraction: 0.67)
Global model successfully updated with aggregated weights.

Final Aggregated Global Weights (first 3 values): [-0.1738981306552887, 0.2657092213630676, 0.25094518065452576]
